# Deep Learning Fashion MNIST


## Libraries

In [ ]:
# installing keras-tuner
!pip install keras-tuner --upgrade

In [ ]:
import gc
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import os
import seaborn as sns
import keras_tuner as kt
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

from sklearn.metrics import confusion_matrix
from tensorflow.keras import backend as K # Importing Keras backend (by default it is Tensorflow)
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import Input, Conv2D, Dense, Dropout, Flatten, MaxPool2D , BatchNormalization # Layers to be used for building our model
from tensorflow.keras.models import Model # The class used to create a model
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.random import set_seed
from sklearn.metrics import accuracy_score, classification_report, precision_recall_curve, auc, f1_score
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint
from keras.callbacks import ReduceLROnPlateau, EarlyStopping
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import train_test_split

## Define the Function for Learning Curves

In [ ]:
# define the function for plotting the learning curves
def plot_history(hs, epochs, metric):
    print()
    fig = plt.figure(figsize=(9, 7))
    plt.style.use('dark_background')
    plt.rcParams['figure.figsize'] = [15, 8]
    plt.rcParams['font.size'] = 10
    plt.clf()
    for label in hs:
        plt.plot(hs[label].history[metric], label='{0:s} train {1:s}'.format(label, metric), linewidth=2)
        plt.plot(hs[label].history['val_{0:s}'.format(metric)], label='{0:s} validation {1:s}'.format(label, metric), linewidth=2)
    x_ticks = np.arange(0, epochs + 1,4)
    x_ticks [0] += 1
    plt.xticks(x_ticks)
    plt.ylim((0, 1))
    plt.xlabel('Epochs')
    plt.ylabel('Loss' if metric=='loss' else 'Accuracy')
    plt.legend()
    plt.show()

## Clean up Function

In [ ]:
# define the function for cleaning up
def clean_up(model):
    K.clear_session()
    del model
    gc.collect()

##Report Function

In [ ]:
# define the report function
def report(history_model, model_eval, model):

  print("Train Loss     : {0:.5f}".format(history_model.history['loss'][-1]))
  print("Validation Loss: {0:.5f}".format(history_model.history['val_loss'][-1]))
  print("Test Loss      : {0:.5f}".format(model_eval[0]))
  print("---")
  print("Train Accuracy     : {0:.5f}".format(history_model.history['accuracy'][-1]))
  print("Validation Accuracy: {0:.5f}".format(history_model.history['val_accuracy'][-1]))
  print("Test Accuracy      : {0:.5f}".format(model_eval[1]))

  print(f"\nPrecision, recall, F1 scores for each class (train set)")
  print(classification_report(Y_train, np.round(model.predict(X_train)).astype("int32")))
  print(f"\nPrecision, recall, F1 scores for each class (validation set)")
  print(classification_report(Y_dev, np.round(model.predict(X_dev)).astype("int32")))
  print(f"\nPrecision, recall, F1 scores for each class (test set)")
  print(classification_report(Y_test, np.round(model.predict(X_test)).astype("int32")))

  predictions = model.predict(X_test)
  y_pred = np.argmax(predictions, axis= 1)
  y_true=np.argmax(Y_test, axis= 1)
  cm = confusion_matrix(y_true, y_pred)
  labels = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
  class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

  fig = plt.figure(figsize=(8,6))
  ax= plt.subplot()
  sns.heatmap(cm, annot=True, ax = ax, fmt = 'g')
  ax.set_xlabel('Predicted', fontsize=20)
  ax.xaxis.set_label_position('bottom')
  plt.xticks(rotation=90)
  ax.xaxis.set_ticklabels(class_names, fontsize = 10)
  ax.xaxis.tick_bottom()
  ax.set_ylabel('True', fontsize=20)
  ax.yaxis.set_ticklabels(class_names, fontsize = 10)
  plt.yticks(rotation=0)
  plt.title('Confusion Matrix Plot', fontsize=20)
  plt.show()

## MLP


#### Data Importing & Preprocessing
**Task**: The task at hand is to identify the type of fashion item given an image representation.

**Data**:
- The dataset consists of grayscale images of fashion items, each measuring 28x28 pixels.
- To use a Multi-Layer Perceptron (MLP) for classification, we need to reshape the images into a flattened format.
- The pixel values in the dataset range from 0 to 255, so we apply normalization to scale them down to the range of 0 to 1.

In [ ]:
# loading the data
classes = 10
(X_train_dev, y_train_dev), (X_test, y_test) = fashion_mnist.load_data()

# splitting the data
X_train, X_dev, y_train, y_dev = train_test_split(X_train_dev, y_train_dev, test_size=0.1, random_state=7)

# print the results
print("Train Images shape:", X_train.shape)
print("Test Images shape:", X_test.shape)

# Change the label names
Label_name_dict = {
 0: "T-shirt/top",
 1: "Trouser",
 2: "Pullover",
 3: "Dress",
 4: "Coat",
 5: "Sandal",
 6: "Shirt",
 7: "Sneaker",
 8: "Bag",
 9: "Ankle boot"
}

In [ ]:
# print a sample of images
plt.imshow(X_train[2])

# print the corresponding class
print(Label_name_dict[y_train[2]])

In [ ]:
# print a sample of images
plt.imshow(X_train[7])

# print the corresponding class
print(Label_name_dict[y_train[7]])

In [ ]:
# Print the number of images per category per data set

print("For train (before splitting)\n"+"----------------------------")

for i in np.unique(y_train_dev,return_counts=True)[0]:
  print(Label_name_dict[i], "count:",np.unique(y_train_dev,return_counts=True)[1][i])
print("")

print("For train set\n"+"-------------")

for i in np.unique(y_train,return_counts=True)[0]:
  print(Label_name_dict[i], "count:",np.unique(y_train,return_counts=True)[1][i])
print("")

print("For validation set\n"+"------------------")
for i in np.unique(y_dev,return_counts=True)[0]:
  print(Label_name_dict[i], "count:",np.unique(y_dev,return_counts=True)[1][i])
print("")

print("For test set\n"+"------------")
for i in np.unique(y_test,return_counts=True)[0]:
  print(Label_name_dict[i], "count:",np.unique(y_test,return_counts=True)[1][i])

#### Data Flattening

In [ ]:
# reshaping the data sets
X_train = X_train.reshape(54000, 784)
X_dev = X_dev.reshape(6000, 784)
X_test = X_test.reshape(10000, 784)
X_train_dev = X_train_dev.reshape(60000, 784)

# changing the data types
X_train = X_train.astype('float32')
X_dev = X_dev.astype('float32')
X_test = X_test.astype('float32')
X_train_dev = X_train_dev.astype('float32')

# normalizatio
X_train /= 255
X_dev /= 255
X_test /= 255
X_train_dev /= 255

# creating one hot veftors for the classes
Y_train = to_categorical(y_train, classes)
Y_dev = to_categorical(y_dev, classes)
Y_test = to_categorical(y_test, classes)
Y_train_dev = to_categorical(y_train_dev, classes)

#### MLP creation
1. We create an MLP as a class in order to pass the optimizer as an argument later as well as passing a boolean argument for the dropout layers.
2. In the architecture of our functional MLP we hypertune:
    - The number of hidden layers, their activation function, the number of units and their kernel initializer.
    - The number of the dropout rate of every hidden layer.
    - The value of the learning rate.
    - The value of the batch size.
    - The kernel initializer of the output layer.


In [ ]:
# Creating a class for MLP tuning
class MyHyperModel(kt.HyperModel):
    # users' initial values
    def __init__(self,layer_dropout,optimizer):
        self.layer_dropout = layer_dropout
        self.optimizer = optimizer

    # Define the MLP achitecture
    def build(self, hp):
        np.random.seed(1966) # Define the seed for numpy to have reproducible experiments.
        set_seed(1908) # Define the seed for Tensorflow to have reproducible experiments.

        # Define the input layer.
        input = Input(
            shape=(784,),
            name='Input'
        )

        x = input
        # Define the remaining hidden layers.
        for i in range(hp.Int('hidden layers',2,5)):
            x = Dense(
                 units=hp.Choice('units of Dense-{0:d}'.format(i + 1),[128,256,512]),
                 kernel_initializer=hp.Choice('kernel_initializer',['glorot_uniform','glorot_normal']), # weights initialization
                 activation=hp.Choice('activation_function',['swish','tanh',"gelu"]),
                 name='Hidden-{0:d}'.format(i + 1)
            )(x)

            # Dropout (TRUE or FALSE)
            if self.layer_dropout:
               x = Dropout(
                    rate=hp.Float('dropout_rate-{0:d}'.format(i + 1),0,0.6,step=0.2),
                    name='Dropout-{0:d}'.format(i + 1))(x)

        # Define the output layer.
        output = Dense(
            units=classes,
            kernel_initializer=hp.Choice('kernel_initializer',['glorot_uniform','glorot_normal',"he_normal"]),
            activation=hp.Choice('output_activation_function',['softmax']),
            name='Output'
        )(x)

        # Define the model and train it.
        model = Model(inputs=input, outputs=output)
        model.compile(optimizer=self.optimizer(learning_rate=hp.Float("lr",0.00001,0.001)), loss='categorical_crossentropy', metrics=['accuracy'])
        return model
    # Batch size tuning
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            *args,
            batch_size=hp.Choice("batch_size", [16, 32,64,128,256]),
            **kwargs,
        )

#### MLP Using Adam as Optimizer


In [ ]:
# define the tuner
tuner_adam = kt.BayesianOptimization( # using bayesian optimizer
    MyHyperModel(True,Adam),
    objective="val_accuracy",
    max_trials=10,
    overwrite=True
)

In [ ]:
# searching for the best model
tuner_adam.search(X_train, Y_train, epochs=100, callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=2, cooldown=2),tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5,restore_best_weights=True)],
                  validation_data=(X_dev,Y_dev))

In [ ]:
# summary of the best model
tuner_adam.results_summary(1)

In [ ]:
# define the model with the best parameters after tuning process
best_hps_mlp_adam = tuner_adam.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the optimal hyperparameters and train it on the data for 100 epochs
model_adam_mlp = tuner_adam.hypermodel.build(best_hps_mlp_adam)
#model_adam_mlp.save("adam_mlp_model")
history_adam_mlp = model_adam_mlp.fit(X_train, Y_train, epochs=100,batch_size=32, callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=2, cooldown=2),tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5,restore_best_weights=True)], validation_data=(X_dev,Y_dev))


In [ ]:
# It can be used to reconstruct the model identically.
#model_adam_mlp = keras.models.load_model("adam_mlp_model")
# summary of the best model
model_adam_mlp.summary()
#history_adam_mlp = model_adam_mlp.fit(X_train, Y_train, epochs=50,batch_size=128,callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=2, cooldown=2),tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5,restore_best_weights=True)], validation_data=(X_val,Y_val))

In [ ]:
val_acc_per_epoch_mlp_adam = history_adam_mlp.history['val_accuracy']
best_epoch_mlp_adam = val_acc_per_epoch_mlp_adam.index(max(val_acc_per_epoch_mlp_adam)) + 1
print('Best epoch: %d' % (best_epoch_mlp_adam,))

In [ ]:
# evaluation of the final model
model_adam_mlp_eval = model_adam_mlp.evaluate(X_test, Y_test)

# cleaning up
clean_up(model=model_adam_mlp)

#### MLP Using Adam | Results
- We developed an MLP using the Adam optimizer. In general, Adam's algorithm is obvious that it perfoms better than the respective MLP attempt that uses the SGD optimizer.
  

In [ ]:
# Plot train and validation error per epoch.9+
plot_history(hs={'MLP With Adam': history_adam_mlp}, epochs=best_epoch_mlp_adam, metric='loss')
plot_history(hs={'MLP With Adam': history_adam_mlp}, epochs=best_epoch_mlp_adam, metric='accuracy')

**Learning curves**
- On the first plot we can see that the training loss curve is stabilizing after some epochs. Also, the validation loss curve seem to have more ups and downs but in the end it is consistently above the training one.
- Similar results can be seen in the second plot.
- All in all the two lines are close and the model has an relatively good overall performance.

In [ ]:
# printing the classifcation report
report(history_adam_mlp, model_adam_mlp_eval, model_adam_mlp)

**Classification Report**
- Here we can observe the results more precisely for each category.
- Generally, we observe that some categories are more separable than others (e.g., Sandals), which makes sense since the algorithm finds it easier to distinguish a sandal from a t-shirt compared to a t-shirt and a pullover that may have similar shapes.

####MLP Using SGD as Optimizer


In [ ]:
# define the tuner
tuner_sgd = kt.BayesianOptimization(
    MyHyperModel(True, SGD),
    objective="val_accuracy",
    max_trials=10,
    overwrite=True
)

In [ ]:
# searching for the best model
tuner_sgd.search(X_train, Y_train, epochs=100, callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=2, cooldown=2),tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5,restore_best_weights=True)],validation_data=(X_dev,Y_dev))

In [ ]:
# summay of the best model
tuner_sgd.results_summary(1)

In [ ]:
# define the model with the best parameters after tuning process
best_hps_mlp_sgd = tuner_sgd.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the optimal hyperparameters and train it on the data for 50 epochs
model_sgd_mlp = tuner_sgd.hypermodel.build(best_hps_mlp_sgd)

history_sgd_mlp = model_sgd_mlp.fit(X_train, Y_train, epochs=100, batch_size = 16, callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=2, cooldown=2),tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5,restore_best_weights=True)], validation_data=(X_dev,Y_dev))

In [ ]:
# It can be used to reconstruct the model identically.
#model_sgd_mlp = keras.models.load_model("sgd_mlp_model")
model_sgd_mlp.summary()
#history_sgd_mlp = model_sgd_mlp.fit(X_train, Y_train, epochs=50, batch_size = 32, callbacks=[ReduceLROnPlateau(monitor='val_loss', patience=2, cooldown=2),tf.keras.callbacks.EarlyStopping('val_loss', patience=5,restore_best_weights=True)], validation_data=(X_dev,Y_dev))

In [ ]:
val_acc_per_epoch_mlp_sgd = history_sgd_mlp.history['val_accuracy']
best_epoch_mlp_sgd = val_acc_per_epoch_mlp_sgd.index(max(val_acc_per_epoch_mlp_sgd)) + 1
print('Best epoch: %d' % (best_epoch_mlp_sgd,))

In [ ]:
# evaluation of the final model
eval_result_mlp_sgd = model_sgd_mlp.evaluate(X_test, Y_test)

# cleaning up
clean_up(model_sgd_mlp)

#### MLP Using SGD | Results
- We create an MLP using the SGD optimizer:
   - It does not perfom as good as our MLP with the Adam Optimizer.
   - We could have tried to improve our model by using more epochs as it seems that it does not converge during the first 100 epochs.
   - We could also have tried to improve the perfomance of our MLP model by adding more trials in our tuner search, more options in the activation functions and kernel initializer and by trying other opimizers, e.g., AdamW optimizer, which can combine the convergence benefits of both Adam and SGD in theory.

In [ ]:
# Plot train and validation error per epoch.
plot_history(hs={'MLP With SGD': history_sgd_mlp}, epochs=best_epoch_mlp_sgd, metric='loss')
plot_history(hs={'MLP With SGD': history_sgd_mlp}, epochs=best_epoch_mlp_sgd, metric='accuracy')

In [ ]:
# printing the classification report
report(history_sgd_mlp, eval_result_mlp_sgd, model_sgd_mlp)

**Classification Report**
- It is evident here that this model is certainly worse than the Adam MLP. Again it can be predict some categories well, like the Bag , but is is even worse in difficult categories like Shirt.
- More specifically there are many pullovers, Tshirt/tops and even Coats that are missclassified as Shirts as well as a lot of Shirts that are missclassified as the aforementioned categories.
- Furthermore there is another problem with the predictions that is not as evident as the previous one. That is the fact that some sandals, sneakers and ankle boots are geting missclassified. These predictions are not as problematic as the ones concern Shirts, but they are certainly worse than the adam MLP's.

## CNN

### Data Preprocessing
**Data**:
- The data are grayscale 28*28 images of fashion items
- We will use a CNN to classify them (no need to flatten as when using an MLP)
- The values of the inputs are in [0, 255] so we normalize them to [0, 1]

In [ ]:
# loading the data
classes = 10
(X_train_val, y_train_val), (X_test, y_test) = fashion_mnist.load_data()

# splitting the data
X_train, X_dev, y_train, y_dev = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=1888)

# reshaping the data
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_dev = X_dev.reshape(X_dev.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)
X_train_val = X_train_val.reshape(X_train_val.shape[0], 28, 28, 1)

# changing the type of data set
X_train = X_train.astype('float32')
X_dev = X_dev.astype('float32')
X_test = X_test.astype('float32')
X_train_val = X_train_val.astype('float32')

# normilize the data
X_train /= 255
X_dev /= 255
X_test /= 255
X_train_val /= 255

# creating tone-hot vectors for labeled data
Y_train = to_categorical(y_train, classes)
Y_dev = to_categorical(y_dev, classes)
Y_test = to_categorical(y_test, classes)
Y_train_val = to_categorical(y_train_val, classes)

###CNN Creation

- We create a CNN as a class in order to pass the optimizer as an argument later as well as passing a boolean argument for the dropout layers. Also we can later control if we use Batch Normalization through a boolean argument.
- We use a Functional Model.
- In the architecture of our functional CNN we hypertune:
    - The number of hidden layers, their activation function the units number and the kernel initializer.
    - The dropout rate of every hidden layer.
    - The value of the learning rate.
    - The value of the batch size.
    - The kernel initializer of the output layer
- The basic architecture of our CNN was not hypertuned, e.g. , filters, kernel size, pool side etc.

In [ ]:
# Creating a class for CNN tuning
class MyCNNHyperModel(kt.HyperModel):
      # users' initial values
    def __init__(self, conv_dropout, optimizer, batch_norm):
        self.conv_dropout = conv_dropout
        self.optimizer = optimizer
        self.batch_norm = batch_norm
    # Define the CNN achitecture
    def build(self, hp):
        np.random.seed(1966)  # Define the seed for numpy to have reproducible experiments.
        set_seed(7)  # Define the seed for Tensorflow to have reproducible experiments.

        # Define the input layer.
        input = Input(shape=(28, 28, 1), name='Input')

        x = input
        # Define the convolutional layers.
        for i in range(hp.Int('convolutional_layers', 1, 3)):
            x = Conv2D(
                filters=8 * (2 ** i),
                kernel_size=(3, 3),
                strides=(1, 1),
                padding='same',
                dilation_rate=(1, 1),
                activation=hp.Choice('activation_function', ['swish', 'tanh', 'gelu']),
                kernel_initializer=hp.Choice('kernel_initializer', ['glorot_uniform', 'glorot_normal']),
                name='Conv2D-{0:d}-a'.format(i + 1)
            )(x)
            x = Conv2D(
                filters=8 * (2 ** i),
                kernel_size=(3, 3),
                strides=(1, 1),
                padding='same',
                dilation_rate=(1, 1),
                activation=hp.Choice('activation_function', ['swish', 'tanh', 'gelu']),
                kernel_initializer=hp.Choice('kernel_initializer', ['glorot_uniform', 'glorot_normal']),
                name='Conv2D-{0:d}-b'.format(i + 1)
            )(x)

            if self.batch_norm:
                x = BatchNormalization()(x)
            # Define the max-pooling layers.
            x = MaxPool2D(
                pool_size=(2, 2),
                strides=(2, 2),  # to change
                padding='same',
                name='MaxPool2D-{0:d}'.format(i + 1)
            )(x)
            # Dropout (TRUE or FALSE)
            if self.conv_dropout:
                x = Dropout(
                    rate=hp.Float('dropout_rate-{0:d}'.format(i + 1), 0, 0.6, step=0.2),
                    name='Dropout-{0:d}'.format(i + 1)
                )(x)

        # Flatten the convolved images so as to input them to a Dense Layer
        x = Flatten(name='Flatten')(x)

        # Define the output layer.
        output = Dense(
            units=classes,
            activation=hp.Choice('output_activation_function', ['softmax']),
            name='Output'
        )(x)

        # Define the model and compile it.
        model = Model(inputs=input, outputs=output)
        model.compile(optimizer=self.optimizer(learning_rate=hp.Float("lr", 0.00001, 0.001)),
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])
        return model
    # Batch size tuning
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            *args,
            batch_size=hp.Choice("batch_size", [16, 32, 64, 128, 256]),
            **kwargs,
        )

###Using Adam with Batch Normalization



In [ ]:
# define the tuner
tuner_cnn_adam_with_Batch = kt.BayesianOptimization(
    MyCNNHyperModel(True,Adam,True),
    objective="val_accuracy",
    max_trials=10,
    overwrite=True
)

In [ ]:
# searching for the best model
tuner_cnn_adam_with_Batch.search(X_train, Y_train, epochs=100, callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=5, cooldown=2),
                                                                         tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5, restore_best_weights=True)],
                                                                         validation_data=(X_dev,Y_dev))

In [ ]:
# summary of the best model
tuner_cnn_adam_with_Batch.results_summary(1)

In [ ]:
# define the model with the best parameters after tuning process
best_hps_cnn_adam_with_Batch = tuner_cnn_adam_with_Batch.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the optimal hyperparameters and train it on the data for 100 epochs
model_adam_cnn_with_Batch = tuner_cnn_adam_with_Batch.hypermodel.build(best_hps_cnn_adam_with_Batch)

#model_adam_cnn_with_Batch.save("adam_cnn_model_with_Batch")
history_adam_cnn_with_Batch = model_adam_cnn_with_Batch.fit(X_train, Y_train, epochs=100,batch_size = 16 ,callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=2, cooldown=2),tf.keras.callbacks.EarlyStopping('val_loss', patience=5,restore_best_weights=True)], validation_data=(X_dev,Y_dev))

In [ ]:
# summary of the best model
model_adam_cnn_with_Batch.summary()

In [ ]:
val_acc_per_epoch_cnn_adam_with_Batch = history_adam_cnn_with_Batch.history['val_accuracy']
best_epoch_cnn_adam_with_Batch = val_acc_per_epoch_cnn_adam_with_Batch.index(max(val_acc_per_epoch_cnn_adam_with_Batch)) + 1
print('Best epoch: %d' % (best_epoch_cnn_adam_with_Batch,))

In [ ]:
# evaluation of the final model
eval_result_cnn_adam_with_Batch = model_adam_cnn_with_Batch.evaluate(X_test, Y_test)

# cleaning up
clean_up(hypermodel_cnn_adam_with_Batch)

### Results
- Firstly, we create a CNN using Adam optimizer. In general, for our CNN's we only used Adam Optimizer, since in comparison with other optimizers has better performance as we saw in our first approach on MLPs.
   - We also add in this CNN Batch Normalization. We observed that without batch normilization our model seems to have **slightly**  better performance on the test set.

In [ ]:
# Plot train and validation error per epoch.
plot_history(hs={'CNN With Adam and Batch Normalization': history_adam_cnn_with_Batch}, epochs=best_epoch_cnn_adam_with_Batch, metric='loss')
plot_history(hs={'CNN With Adam and Batch Normalization': history_adam_cnn_with_Batch}, epochs=best_epoch_cnn_adam_with_Batch, metric='accuracy')

**Learning curves**
- Here we can also see that the train curve flattens as the epochs increase and the validation curve is again slightly better than the train one. This likely happens due to the fact that we used batch normalization, as this is not happens in our next approach where using no batch normalization.

In [ ]:
# reporting
report(history_adam_cnn_with_Batch, eval_result_cnn_adam_with_Batch, model_adam_cnn_with_Batch)

**Classification Report**
-Regarding the performance of the CNN using Batch Normalization, we can see that the results are similar to the Adam's MLP, meaning that again T-shirts, shirts and coats predictions are getting mixed. But the overall test acuracy is consistently higher 91% .

###Using Adam without Batch Normalization

In [ ]:
# define the tuner
tuner_cnn_adam_No_Batch = kt.BayesianOptimization(
    MyCNNHyperModel(True,Adam,False),
    objective="val_accuracy",
    max_trials=10,
    overwrite=True
)

In [ ]:
# searching for the best model
tuner_cnn_adam_No_Batch.search(X_train, Y_train, epochs=100,
                               callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=2, cooldown=2),
                                          tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5,restore_best_weights=True)],
                               validation_data=(X_dev,Y_dev))

In [ ]:
# summay of the best model
tuner_cnn_adam_No_Batch.results_summary(1)

In [ ]:
# define the model with the best parameters after tuning process
best_hps_cnn_adam_No_Batch = tuner_cnn_adam_No_Batch.get_best_hyperparameters(num_trials=1)[0]
# Build the model with the optimal hyperparameters and train it on the data for 50 epochs
model_adam_cnn_No_Batch = tuner_cnn_adam_No_Batch.hypermodel.build(best_hps_cnn_adam_No_Batch)
#model_adam_cnn_No_Batch.save("adam_cnn_model_No_Batch")
history_adam_cnn_No_Batch = model_adam_cnn_No_Batch.fit(X_train, Y_train, batch_size= 16 ,epochs=100,
                                                        callbacks=[ReduceLROnPlateau(monitor='val_accuracy', patience=2, cooldown=2),
                                                                   tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5,restore_best_weights=True)],
                                                        validation_data=(X_dev,Y_dev))

In [ ]:
# summary of the best model
model_adam_cnn_No_Batch.summary()

In [ ]:
val_acc_per_epoch_cnn_adam_No_Batch = history_adam_cnn_No_Batch.history['val_accuracy']
best_epoch_cnn_adam_No_Batch = val_acc_per_epoch_cnn_adam_No_Batch.index(max(val_acc_per_epoch_cnn_adam_No_Batch)) + 1
print('Best epoch: %d' % (best_epoch_cnn_adam_No_Batch,))

In [ ]:
# evaluation of the final model
eval_result_cnn_adam_No_Batch = model_adam_cnn_No_Batch.evaluate(X_test, Y_test)

# cleaning up
clean_up(model_adam_cnn_No_Batch)

### Results

In [ ]:
# Plot train and validation error per epoch.
plot_history(hs={'CNN With Adam and Without Batch Normalization': history_adam_cnn_No_Batch}, epochs=best_epoch_cnn_adam_No_Batch, metric='loss')
plot_history(hs={'CNN With Adam and Without Batch Normalization': history_adam_cnn_No_Batch}, epochs=best_epoch_cnn_adam_No_Batch, metric='accuracy')

**Learning curves**
- Here we can see that both line stabilize and are very close, with the train loss being lower than the validation during later epochs.
- A similar conclusion can be seen on the second plot where the train accuracy is slightly higher than the validation one.

In [ ]:
# printing the classification report
report(history_adam_cnn_No_Batch, eval_result_cnn_adam_No_Batch, model_adam_cnn_No_Batch)

**Classification report:**
The CNN without batch normalization performs almost identically as the earlier one (with batch normalization). Overall, we cannot say with certainty that the differences between the 2 CNN's predictions are significant.


Through various attempts we can safely assume that the most important things to tune (which had a significant difference to our scores) were the batch size and the learning rate as well as the optimizer. As a result we would like to try different values for the first two and also different optimizers to try solve the problem with better results.